In [120]:
import sys
from PyQt6 import QtCore, QtGui, QtWidgets, uic


In [121]:
# Sub-windows
class AddWindow(QtWidgets.QMainWindow):

  # Constructor Method
  def __init__(self, parent=None):
    super().__init__()
    self.parent = parent

    # Load UI
    uic.loadUi("add_poll_ui.ui", self)

    # Windows Settgins
    self.setWindowTitle("Add new Poll")
    self.setWindowIcon(
      QtGui.QIcon("logo.png")
    )

    # Check total polls
    self.totalPoll = len(parent.pollList)

    self.pollIdLineEdit = self.findChild(QtWidgets.QLineEdit, "pollIdLineEdit")
    self.titleLineEdit = self.findChild(QtWidgets.QLineEdit, "titleLineEdit")
    self.descriptionLineEdit = self.findChild(QtWidgets.QLineEdit, "descriptionLineEdit")
    self.authorLineEdit = self.findChild(QtWidgets.QLineEdit, "authorLineEdit")
    self.createdAtLineEdit = self.findChild(QtWidgets.QLineEdit, "createdAtLineEdit")
    self.updatedAtLineEdit = self.findChild(QtWidgets.QLineEdit, "updatedAtLineEdit")

    self.addButton = self.findChild(QtWidgets.QPushButton, "addButton")
    self.cancelButton = self.findChild(QtWidgets.QPushButton, "cancelButton")

    # Button events
    self.cancelButton.clicked.connect(self.closeWindows)
    self.addButton.clicked.connect(self.addPoll)

  # Functions
  def closeWindows(self):
    self.close()

  def addPoll(self):

    title = self.titleLineEdit.text()
    description = self.descriptionLineEdit.text()
    author = self.authorLineEdit.text()
    createdAt = self.createdAtLineEdit.text()
    updatedAt = self.updatedAtLineEdit.text()

    # Validate: All filed must not empty
    if not title or not description or not author or not createdAt or not updatedAt:
      # Notify user to fill all inputs
      QtWidgets.QMessageBox.warning(self, "Warning!", "Please fill all the inputs")
      return

    # Make the ID unique (Key)
    pollId = title

    newPoll = {
      "id": pollId,
      "title": title,
      "description": description,
      "author": author,
      "createdAt": createdAt,
      "updatedAt": updatedAt
    }

    # Confirm dialog
    confirmDialog = QtWidgets.QMessageBox()
    confirmDialog.setWindowTitle("Confirmation")
    confirmDialog.setText("Add new poll?")
    confirmDialog.setIcon(confirmDialog.Icon.Warning)
    confirmDialog.setStandardButtons(
      confirmDialog.StandardButton.Ok | confirmDialog.StandardButton.Cancel
    )

    isConfirmed = confirmDialog.exec()

    # If user press "Ok" button
    if isConfirmed == confirmDialog.StandardButton.Ok:
      # Execution: Add new poll into MainApp (parents)
      self.parent.pollList.append(newPoll)
      print("[Added]")
      self.parent.refreshListWidget()
      self.close()

In [122]:
class DetailWindows(QtWidgets.QMainWindow):

  def __init__(self, parent = None, detailIndex=-1):
    super().__init__()
    self.parent = parent

    # Load UI
    uic.loadUi("poll_details_ui.ui", self)

    # Windows Settgins
    self.setWindowTitle("Poll details")
    self.setWindowIcon(
      QtGui.QIcon("logo.png")
    )

    # UI Define
    self.closeButton = self.findChild(QtWidgets.QPushButton, "closeButton")
    self.pollIdLineEdit = self.findChild(QtWidgets.QLineEdit, "pollIdLineEdit")
    self.titleLineEdit = self.findChild(QtWidgets.QLineEdit, "titleLineEdit")
    self.descriptionLineEdit = self.findChild(QtWidgets.QLineEdit, "descriptionLineEdit")
    self.authorLineEdit = self.findChild(QtWidgets.QLineEdit, "authorLineEdit")
    self.createdAtLineEdit = self.findChild(QtWidgets.QLineEdit, "createdAtLineEdit")
    self.updatedAtLineEdit = self.findChild(QtWidgets.QLineEdit, "updatedAtLineEdit")

    # Initial value
    pollDetail = parent.pollList[detailIndex]

    self.pollIdLineEdit.setText(pollDetail["id"])
    self.titleLineEdit.setText(pollDetail["title"])
    self.descriptionLineEdit.setText(pollDetail["description"])
    self.authorLineEdit.setText(pollDetail["author"])
    self.createdAtLineEdit.setText(pollDetail["createdAt"])
    self.updatedAtLineEdit.setText(pollDetail["updatedAt"])

    # Button events
    self.closeButton.clicked.connect(self.closeWindows)
  
  # Functions
  def closeWindows(self):
    self.close()

In [123]:
class EditWindows(QtWidgets.QMainWindow):

  def __init__(self, parent = None, detailIndex=-1):
    super().__init__()
    self.parent = parent
    self.detailIndex = detailIndex

    # Load UI
    uic.loadUi("edit_poll_ui.ui", self)

    # Windows Settgins
    self.setWindowTitle("Edit poll")
    self.setWindowIcon(
      QtGui.QIcon("logo.png")
    )

    # UI Define
    self.pollIdLineEdit = self.findChild(QtWidgets.QLineEdit, "pollIdLineEdit")
    self.titleLineEdit = self.findChild(QtWidgets.QLineEdit, "titleLineEdit")
    self.descriptionLineEdit = self.findChild(QtWidgets.QLineEdit, "descriptionLineEdit")
    self.authorLineEdit = self.findChild(QtWidgets.QLineEdit, "authorLineEdit")
    self.createdAtLineEdit = self.findChild(QtWidgets.QLineEdit, "createdAtLineEdit")
    self.updatedAtLineEdit = self.findChild(QtWidgets.QLineEdit, "updatedAtLineEdit")

    self.updateButton = self.findChild(QtWidgets.QPushButton, "updateButton")
    self.cancelButton = self.findChild(QtWidgets.QPushButton, "cancelButton")

    # Initial value
    pollDetail = parent.pollList[detailIndex]
    self.pollIdLineEdit.setText(pollDetail["id"])
    self.titleLineEdit.setText(pollDetail["title"])
    self.descriptionLineEdit.setText(pollDetail["description"])
    self.authorLineEdit.setText(pollDetail["author"])
    self.createdAtLineEdit.setText(pollDetail["createdAt"])
    self.updatedAtLineEdit.setText(pollDetail["updatedAt"])

    # Make the ID unique (Key) [Fixed - cannot changed]
    self.defaultPollId = self.pollIdLineEdit.text()

    # Button events
    self.updateButton.clicked.connect(self.editPoll)
    self.cancelButton.clicked.connect(self.closeWindows)
  
  # Functions
  def closeWindows(self):
    self.close()

  def editPoll(self):
    title = self.titleLineEdit.text()
    description = self.descriptionLineEdit.text()
    author = self.authorLineEdit.text()
    createdAt = self.createdAtLineEdit.text()
    updatedAt = self.updatedAtLineEdit.text()

    # Validate: All filed must not empty
    if not title or not description or not author or not createdAt or not updatedAt:
      # Notify user to fill all inputs
      QtWidgets.QMessageBox.warning(self, "Warning!", "Please fill all the inputs")
      return

    edtiedPoll = {
      "id": self.defaultPollId,
      "title": title,
      "description": description,
      "author": author,
      "createdAt": createdAt,
      "updatedAt": updatedAt
    }

    # Confirm dialog
    confirmDialog = QtWidgets.QMessageBox()
    confirmDialog.setWindowTitle("Confirmation")
    confirmDialog.setText("Edit poll?")
    confirmDialog.setIcon(confirmDialog.Icon.Warning)
    confirmDialog.setStandardButtons(
      confirmDialog.StandardButton.Ok | confirmDialog.StandardButton.Cancel
    )

    isConfirmed = confirmDialog.exec()

    # If user press "Ok" button
    if isConfirmed == confirmDialog.StandardButton.Ok:
      # Execution: Edit poll data into MainApp (parents)
      self.parent.pollList[self.detailIndex] = edtiedPoll
      print("[Edited]")
      self.parent.refreshListWidget()
      QtWidgets.QMessageBox.information(self, "Edit", "Poll edited!")
      self.close()

In [124]:
# Main-windows (Open first when app is running)
class MainWindow(QtWidgets.QMainWindow):

  def __init__(self):
    super().__init__()
    uic.loadUi("poll_ui.ui", self)

    # Windows Settings
    self.setWindowFlag(
      QtCore.Qt.WindowType.WindowCloseButtonHint
    )
    self.setWindowTitle("Poll App")

    # App icon
    self.setWindowIcon(
      QtGui.QIcon("logo.png")
    )

    # App variables (Globals)
    self.pollList = []

    # Define UI components (variables)
    self.pollListWidget = self.findChild(QtWidgets.QListWidget, "pollListWidget")
    self.addButton = self.findChild(QtWidgets.QPushButton, "addButton")
    self.editButton = self.findChild(QtWidgets.QPushButton, "editButton")
    self.deleteButton = self.findChild(QtWidgets.QPushButton, "deleteButton")
    self.detailButton = self.findChild(QtWidgets.QPushButton, "detailButton")

    # Button events
    self.addButton.clicked.connect(self.openAddWindows)
    self.editButton.clicked.connect(self.openEditWindows)
    self.detailButton.clicked.connect(self.openDetailWindows)
    self.deleteButton.clicked.connect(self.deletePoll)

  # Functions
  def openAddWindows(self):
    self.addWindows = AddWindow(self)
    self.addWindows.show()

  def openDetailWindows(self):
    currentItem = self.pollListWidget.currentItem()

    # Validate if poll has selected
    if not currentItem:
      QtWidgets.QMessageBox.warning(self, "Warning!", "No poll seleted!")
      return
    
    currentPollIndex = self.pollListWidget.currentRow()
    self.detailWindows = DetailWindows(self, currentPollIndex)
    self.detailWindows.show()

  def openEditWindows(self):
    currentItem = self.pollListWidget.currentItem()

    # Validate if poll has selected
    if not currentItem:
      QtWidgets.QMessageBox.warning(self, "Warning!", "No poll seleted!")
      return
    
    currentPollIndex = self.pollListWidget.currentRow()
    self.editWindows = EditWindows(self, currentPollIndex)
    self.editWindows.show()

  def deletePoll(self):
    deletedItem = self.pollListWidget.currentItem()

    # If: no item seleted
    if not deletedItem:
      QtWidgets.QMessageBox.warning(self, "Warning!", "No poll seleted")
      return
    
    # Confirm dialog
    confirmDialog = QtWidgets.QMessageBox()
    confirmDialog.setWindowTitle("Confirmation")
    confirmDialog.setIcon(confirmDialog.Icon.Warning)
    confirmDialog.setText("Delete poll?")
    confirmDialog.setStandardButtons(
      confirmDialog.StandardButton.Ok | confirmDialog.StandardButton.Cancel
    )

    isConfirmed = confirmDialog.exec()

    # If user press "Ok" button
    if isConfirmed == confirmDialog.StandardButton.Ok:
      # deletedItemPlaceholder = deletedItem.text()
      deletedIndex = self.pollListWidget.currentRow()

      # Remove data from pollList
      deletedPollData = self.pollList[deletedIndex]
      self.pollList.remove(deletedPollData)

      # Refresh poll list widget
      print("[Deleted]")
      self.refreshListWidget()
      QtWidgets.QMessageBox.information(self, "Deletion", "Poll deleted!")
    
  def refreshListWidget(self):
    # Clear the list widget
    self.pollListWidget.clear()

    # Checking: if poll list > 0 (then refresh)
    if len(self.pollList) > 0:
      for i in range(len(self.pollList)):
        newPollPlanceholder = f"{self.pollList[i]["author"]}: {self.pollList[i]["title"]}"
        # defaultPlaceholder = self.pollList[i]["author"] + ": " + self.pollList[i]["title"]

        # Update the list widget
        self.pollListWidget.addItem(newPollPlanceholder)

    print("Total poll(s):", self.pollList)


In [125]:
app = QtCore.QCoreApplication.instance()
if app is None : app = QtWidgets.QApplication([])
window = MainWindow()
window.show()
app.exec()

[Added]
Total poll(s): [{'id': '1', 'title': '1', 'description': '1', 'author': '1', 'createdAt': '1', 'updatedAt': '1'}]
[Edited]
Total poll(s): [{'id': '1', 'title': '2', 'description': '2', 'author': '2', 'createdAt': '2', 'updatedAt': '2'}]
[Edited]
Total poll(s): [{'id': '1', 'title': '3', 'description': '3', 'author': '3', 'createdAt': '3', 'updatedAt': '3'}]
[Deleted]
Total poll(s): []
[Added]
Total poll(s): [{'id': '1', 'title': '1', 'description': '1', 'author': '1', 'createdAt': '1', 'updatedAt': '1'}]
[Added]
Total poll(s): [{'id': '1', 'title': '1', 'description': '1', 'author': '1', 'createdAt': '1', 'updatedAt': '1'}, {'id': '2', 'title': '2', 'description': '2', 'author': '2', 'createdAt': '2', 'updatedAt': '2'}]
[Added]
Total poll(s): [{'id': '1', 'title': '1', 'description': '1', 'author': '1', 'createdAt': '1', 'updatedAt': '1'}, {'id': '2', 'title': '2', 'description': '2', 'author': '2', 'createdAt': '2', 'updatedAt': '2'}, {'id': '3', 'title': '3', 'description': '

0